# Computer Vision — โจทย์แบบ "หาวัตถุในรูป (กล่อง)" Object Detection

รูป 1 รูป -> หากล่อง (bounding box) + ป้ายของวัตถุ (เช่น หาบ้าน/รถ/ป้ายทะเบียนในรูป)

วิธีในโน้ตบุ๊กนี้: `YOLOv8` (ไลบรารี ultralytics) เป็นวิธีที่ง่ายและแรงสุดสำหรับมือใหม่
สั่งเทรน 2-3 บรรทัดจบ


## เมื่อไหร่ใช้โน้ตบุ๊กนี้

ใช้เมื่อ: โจทย์ให้ทำนาย "ตำแหน่งกล่อง + ชนิดวัตถุ" metric มักเป็น `mAP`
ถ้าโจทย์แค่ "รูปนี้คือคลาสอะไร" (ไม่มีกล่อง) -> กลับไป `image_classification.ipynb` (ง่ายกว่ามาก)

หมายเหตุมือใหม่: detection ยากกว่า classification เพราะข้อมูลต้องมี label เป็นกล่อง (รูปแบบ YOLO)
ถ้าเวลาน้อยและโจทย์เป็น classification ได้ ให้เลือก classification ก่อน

## ขั้นที่ 1 — ติดตั้ง

In [ ]:
!pip -q install ultralytics pandas pyyaml

## ขั้นที่ 2 — ตั้งค่า Kaggle แล้วดาวน์โหลดข้อมูล

ต้องมีไฟล์ `kaggle.json` ก่อน วิธีได้มา: เข้า kaggle.com -> กดรูปโปรไฟล์ -> Settings -> Account -> Create New Token
จะได้ไฟล์ `kaggle.json` หน้าตา `{"username":"...","key":"..."}`

- ถ้ารันบน `Kaggle Notebook`: ข้อมูลอยู่ที่ `/kaggle/input/...` แล้ว ไม่ต้องทำอะไร รันผ่านได้เลย
- ถ้ารันบน `Google Colab / เครื่องตัวเอง`: เอา username กับ key ใส่ในเซลล์ข้างล่าง (จุด `# << แก้`)

In [ ]:
import os, glob, subprocess

COMP = "ใส่-slug-ของ-competition-detection-ตรงนี้"      # << แก้ตรงนี้: slug ของ competition (คือส่วนท้าย URL หลังคำว่า /competitions/)
KAGGLE_USERNAME = ""   # << แก้ (เฉพาะ Colab/เครื่องตัวเอง) เช่น "peetwan1"
KAGGLE_KEY      = ""   # << แก้ (เฉพาะ Colab/เครื่องตัวเอง) คีย์ยาว ๆ จาก kaggle.json

def get_data(comp):
    # ถ้าอยู่บน Kaggle ข้อมูลถูกต่อไว้ให้แล้ว
    kpath = f"/kaggle/input/{comp}"
    if os.path.isdir(kpath):
        print("อยู่บน Kaggle แล้ว ข้อมูลอยู่ที่", kpath); return kpath
    # ถ้าไม่ใช่ Kaggle: เขียน token แล้วโหลดด้วย kaggle CLI
    if KAGGLE_USERNAME and KAGGLE_KEY:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        open(os.path.expanduser("~/.kaggle/kaggle.json"), "w").write(
            '{"username":"%s","key":"%s"}' % (KAGGLE_USERNAME, KAGGLE_KEY))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    os.makedirs("data", exist_ok=True)
    subprocess.run(["kaggle","competitions","download","-c",comp,"-p","data"], check=True)
    for z in glob.glob("data/*.zip"):
        subprocess.run(["unzip","-o","-q",z,"-d","data"], check=True)
    return "data"

DATA_DIR = get_data(COMP)
print("ไฟล์ที่ดาวน์โหลดมา (ดูชื่อไฟล์/โฟลเดอร์ แล้วเอาไปแก้ในเซลล์ถัดไป):")
for p in sorted(glob.glob(os.path.join(DATA_DIR, "**"), recursive=True))[:40]:
    print("  ", p)

## ขั้นที่ 2 — เตรียมข้อมูลแบบ YOLO

YOLO ต้องการ: โฟลเดอร์รูป + ไฟล์ label `.txt` (คลาส x_center y_center w h แบบ normalize 0-1) ต่อรูป 1 ไฟล์
และไฟล์ `data.yaml` ชี้ path รูป train/val + ชื่อคลาส

ถ้า competition ให้ข้อมูลเป็นรูปแบบอื่น (เช่น COCO json หรือ csv ของกล่อง) ต้องแปลงเป็นรูปแบบ YOLO ก่อน
เซลล์ล่างสร้างไฟล์ `data.yaml` ให้ แก้ path/ชื่อคลาสตามจริง

In [ ]:
import yaml, os
# << แก้ path เหล่านี้ให้ตรงกับข้อมูลจริง (โฟลเดอร์ที่มีรูป + label .txt)
data_cfg = {
    "path": os.path.abspath(DATA_DIR),   # โฟลเดอร์หลัก
    "train": "images/train",             # << แก้: โฟลเดอร์รูป train (relative กับ path)
    "val":   "images/val",               # << แก้: โฟลเดอร์รูป val
    "names": {0: "object"},              # << แก้: ชื่อคลาส เช่น {0:"house",1:"car"}
}
with open("data.yaml","w") as f: yaml.safe_dump(data_cfg, f, allow_unicode=True)
print(open("data.yaml").read())

## วิธีที่ 1 — เทรน YOLOv8 (ง่ายสุด)

`yolov8n` = เล็ก/เร็ว, `yolov8s/m` = ใหญ่/แม่นขึ้น ปรับ `epochs`/`imgsz` ตามเวลา/GPU

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")      # << แก้เป็น yolov8s.pt ถ้าอยากแม่นขึ้น
model.train(data="data.yaml", epochs=50, imgsz=640, batch=16)   # << แก้ epochs/imgsz ตามเวลา
metrics = model.val(); print(metrics)

## ขั้นที่ 3 — ทำนาย test + สร้าง submission

รูปแบบ submission ของ detection แตกต่างกันมากในแต่ละ comp (อ่าน sample_submission ให้ดี)
ตัวอย่างล่างสร้างตาราง: id, class, confidence, x_min, y_min, x_max, y_max — แก้ให้ตรงรูปแบบจริง

In [ ]:
import glob, pandas as pd
TEST_IMG_DIR = os.path.join(DATA_DIR, "images/test")   # << แก้ path รูป test
rows = []
for p in sorted(glob.glob(os.path.join(TEST_IMG_DIR, "*"))):
    r = model.predict(p, verbose=False)[0]
    img_id = os.path.splitext(os.path.basename(p))[0]
    for b in r.boxes:
        x1,y1,x2,y2 = b.xyxy[0].tolist()
        rows.append({"id": img_id, "class": int(b.cls[0]), "confidence": float(b.conf[0]),
                     "x_min": x1, "y_min": y1, "x_max": x2, "y_max": y2})
sub = pd.DataFrame(rows)            # << แก้คอลัมน์ให้ตรง sample_submission.csv
sub.to_csv("submission.csv", index=False)
print("บันทึก submission.csv", sub.shape); display(sub.head())

## ขั้นสุดท้าย — ส่ง submission ขึ้น Kaggle

เช็คก่อนส่งทุกครั้ง: ไฟล์ submission ต้องมีชื่อคอลัมน์ + จำนวนแถว ตรงกับ `sample_submission.csv` เป๊ะ ๆ
- บน Kaggle Notebook: กดปุ่ม `Submit` ที่แถบขวา (ง่ายสุด) หรือใช้คำสั่งข้างล่าง
- บน Colab/เครื่อง: รันเซลล์ข้างล่าง (ต้องตั้ง token แล้ว)

In [ ]:
FILE = "submission.csv"   # << แก้เป็นชื่อไฟล์ที่คะแนนดีสุด
!kaggle competitions submit -c {COMP} -f {FILE} -m "yolov8 detection"
!kaggle competitions submissions -c {COMP} | head